# Barred dwarfs with MUSE data: archive query, visual vetting, geometry

Companion notebook to the pipeline in `scripts/`. Order of work:

1. load the star-forming dwarfs and the barred candidates (steps 1-2)
2. **MUSE coverage first**: download the ESO MUSE product lists once and match
   every dwarf locally (barred candidates, or all 1,782 resolvable dwarfs)
3. **visual vetting** of only the MUSE-covered galaxies on Legacy Surveys cutouts -> `products/vetting.csv`
4. optional: Galaxy Zoo DESI bars for the non-SGA dwarfs
5. **disc/bar geometry** and TW suitability for the vetted galaxies
6. TW demo on a synthetic disc, so you know what the real diagram should look like

Run from the repo root (`jupyter notebook notebooks/`), Python >= 3.9.
Sections 2-5 need internet; everything else works offline. Re-running is safe:
archive lists are cached in `data/raw/`, results go to `products/`.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "bartw").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from bartw.config import load_config, data_dir
cfg = load_config()
PROD = ROOT / "products"; PROD.mkdir(exist_ok=True)
print("repo:", ROOT)

## 1. Parent sample and barred candidates

`data/interim/01_sf_dwarfs.parquet` = the paper's 14,786 star-forming dwarfs
with resolution tiers (A: z < 0.012). `products/barred_dwarf_candidates.csv`
= the 66 flagged as barred by the SGA bar catalogue (steps 1-2 must have run).

In [ ]:
sf = pd.read_parquet(data_dir("interim") / "01_sf_dwarfs.parquet")
cand = pd.read_csv(PROD / "barred_dwarf_candidates.csv")
print(f"{len(sf)} star-forming dwarfs; tier A (z<{cfg['desi']['z_max']}): {(sf.Z_TIER.str.startswith('A')).sum()}")
print(f"{len(cand)} barred candidates; {(cand.PRIORITY <= 2).sum()} at z<{cfg['desi']['z_max']}")
display(cand.groupby("PRIORITY").agg(n=("TARGETID", "size"), z_med=("Z", "median"),
        logM_med=("logmstar_h0678", "median"), incl_med=("INCL_DEG", "median"),
        bar_ellip_med=("BAR_ELLIP", "median")).round(3))

## 2. MUSE coverage from the ESO archive

`fetch_all_muse` downloads the complete list of MUSE Phase-3 cubes
(`ivoa.ObsCore`) and raw science frames (`dbo.raw`) once (cached in
`data/raw/eso_muse_all_*.parquet`), then everything is matched locally within
30" (MUSE WFM is 1'x1'; the nucleus must be within ~30" of the pointing).

`SCOPE = "tierA"` matches all 1,782 resolvable dwarfs, not just the barred
candidates: a MUSE-covered dwarf whose bar the SGA catalogue missed (94% of
the dwarfs are not in the SGA at all) is still a target once you see the bar
by eye in section 3. `SCOPE = "candidates"` restricts to the 66.

In [ ]:
from bartw.muse import muse_coverage, summarize_coverage, print_schema
# print_schema(cfg["muse"]["tap_url"], "ivoa.ObsCore"); print_schema(cfg["muse"]["tap_url"], "dbo.raw")  # live column names

SCOPE = "tierA"            # "tierA" (1,782 dwarfs at z<0.012) | "candidates" (66 barred) | "all" (14,786)
if SCOPE == "candidates":
    targets = cand.copy()
elif SCOPE == "tierA":
    targets = sf[sf.Z_TIER.str.startswith("A")].copy()
else:
    targets = sf.copy()
targets = targets.reset_index(drop=True)
print(f"matching {len(targets)} galaxies ({SCOPE}) against the ESO MUSE product lists ...")
hits = muse_coverage(targets.RA.to_numpy(), targets.DEC.to_numpy(), cfg["muse"], cache_dir=data_dir("raw"))
hits.to_parquet(data_dir("interim") / "03_muse_hits.parquet")
cov = summarize_coverage(targets, hits, cfg["muse"])
cov["BARRED_SGA"] = cov.TARGETID.isin(cand.TARGETID)
cov = cov.merge(cand[["TARGETID", "BAR_ELLIP", "BAR_MINUS_DISC_ELLIP", "PRIORITY"]], on="TARGETID", how="left")
has = cov[(cov.N_MUSE_CUBES > 0) | cov.MUSE_RAW_ONLY].copy()
print(f"with MUSE cubes: {int((cov.N_MUSE_CUBES > 0).sum())} | public: {int(cov.MUSE_PUBLIC.sum())} | "
      f">= {cfg['muse']['min_exptime_s']} s: {int((cov.MUSE_DEEP_ENOUGH & (cov.N_MUSE_CUBES > 0)).sum())} | raw only: {int(cov.MUSE_RAW_ONLY.sum())}")
print(f"of which flagged barred by the SGA catalogue: {int(has.BARRED_SGA.sum())}")
cov.to_csv(PROD / "sample_with_muse.csv", index=False)
has = has.sort_values(["BARRED_SGA", "MUSE_EXPTIME_S"], ascending=[False, False]).reset_index(drop=True)
has.to_csv(PROD / "muse_covered_dwarfs.csv", index=False)
cols = ["TARGETID", "sga_galaxy", "RA", "DEC", "Z", "logmstar_h0678", "INCL_DEG", "BARRED_SGA", "BAR_ELLIP",
        "N_MUSE_CUBES", "MUSE_EXPTIME_S", "MUSE_PUBLIC", "MUSE_RAW_ONLY", "MUSE_PROG_IDS"]
display(has[[c for c in cols if c in has]])

In [ ]:
# archive products for one MUSE-covered galaxy (dp_id is what you download from archive.eso.org)
if len(has):
    i = int(np.where(targets.TARGETID.to_numpy() == has.TARGETID.iloc[0])[0][0])
    display(hits[hits.idx == i].drop(columns=["idx"]))

## 3. Visual vetting of the MUSE-covered dwarfs

DR10 grz cutouts (1' on a side, north up, east left; the dashed ellipse is the
SGA R25 with the adopted disc PA). Classify every MUSE-covered galaxy, not only
the SGA-flagged ones:

* `y` clear bar, `w` weak/uncertain bar, `n` no bar, `x` not usable (edge-on, merger,
  companion inside the MUSE field, artefact).

Fill `VET` in the cell after the grids. Saved to `products/vetting.csv`.

In [ ]:
import requests, io
from PIL import Image as PILImage
from matplotlib.patches import Ellipse

def cutout_jpg(ra, dec, size_arcsec=60, layer="ls-dr10", pixscale=0.262):
    size = int(size_arcsec / pixscale)
    url = (f"https://www.legacysurvey.org/viewer/jpeg-cutout?ra={ra}&dec={dec}"
           f"&layer={layer}&pixscale={pixscale}&size={size}")
    r = requests.get(url, timeout=60); r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content)), size

def viewer_link(ra, dec):
    return f"https://www.legacysurvey.org/viewer?ra={ra:.5f}&dec={dec:.5f}&layer=ls-dr10&zoom=14"

def show_grid(df, ncol=5, size_arcsec=60, pixscale=0.262):
    n = len(df); nrow = max(1, int(np.ceil(n / ncol)))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.5 * nrow), squeeze=False)
    for ax, (_, r) in zip(axes.ravel(), df.iterrows()):
        try:
            img, size = cutout_jpg(r.RA, r.DEC, size_arcsec, pixscale=pixscale)
            ax.imshow(img, origin="upper"); c = size / 2
            sma = r.get("sma_sb25", np.nan)
            if np.isfinite(sma) and sma > 0 and np.isfinite(r.get("DISC_BA", np.nan)):
                a = sma / pixscale
                ax.add_patch(Ellipse((c, c), 2 * a, 2 * a * r.DISC_BA, angle=90 + r.DISC_PA, fill=False, ec="w", lw=0.8, ls="--"))
            bar = f" e_bar={r.BAR_ELLIP:.2f}" if np.isfinite(r.get("BAR_ELLIP", np.nan)) else ""
            ax.set_title(f"{int(r.TARGETID)}\nz={r.Z:.4f} i={r.INCL_DEG:.0f}° logM={r.logmstar_h0678:.1f}{bar}\n"
                         f"MUSE {r.MUSE_EXPTIME_S/60:.0f} min {'cube' if r.N_MUSE_CUBES > 0 else 'raw'}", fontsize=7.5)
        except Exception as exc:
            ax.text(0.5, 0.5, f"cutout failed\n{exc}"[:60], ha="center", va="center", fontsize=7, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()

has = pd.read_csv(PROD / "muse_covered_dwarfs.csv") if (PROD / "muse_covered_dwarfs.csv").exists() else has
print(len(has), "MUSE-covered dwarfs to vet. Interactive viewer links:")
display(HTML("<br>".join(f'{int(r.TARGETID)}{" [SGA bar]" if r.BARRED_SGA else ""}: '
                         f'<a href="{viewer_link(r.RA, r.DEC)}" target="_blank">{r.RA:.4f} {r.DEC:+.4f}</a>'
                         for _, r in has.iterrows())))

In [ ]:
show_grid(has.iloc[0:20])      # change the slice for the next batch

In [ ]:
show_grid(has.iloc[20:40])

In [ ]:
# ---- your classifications: TARGETID -> 'y' | 'w' | 'n' | 'x'   (optional note after '#')
VET = {
    # 39627731934251234: "y  # clear bar, cube 60 min, isolated",
}

vet_path = PROD / "vetting.csv"
old = pd.read_csv(vet_path) if vet_path.exists() else pd.DataFrame(columns=["TARGETID", "VET", "VET_NOTE"])
rows = [dict(TARGETID=int(k), VET=str(v).split("#")[0].strip(), VET_NOTE=(str(v).split("#", 1)[1].strip() if "#" in str(v) else ""))
        for k, v in VET.items()]
vet = pd.concat([old[~old.TARGETID.isin([r["TARGETID"] for r in rows])], pd.DataFrame(rows)], ignore_index=True)
vet.to_csv(vet_path, index=False)
print(f"{len(vet)} classifications in {vet_path.name}:", vet.VET.value_counts().to_dict())
has = has.drop(columns=[c for c in ("VET", "VET_NOTE") if c in has]).merge(vet, on="TARGETID", how="left")
print("still to vet:", int(has.VET.isna().sum()))
tw_targets = has[has.VET.isin(["y", "w"])]
print("TW targets so far (y/w):", len(tw_targets))

## 4. Optional: Galaxy Zoo DESI bars for the non-SGA dwarfs

Independent bar indicator for the 94% of dwarfs the SGA catalogue cannot see.
Needs `data/raw/gz_desi_deep_learning_catalog_friendly.parquet` (Zenodo 8360385,
~2 GB; see `scripts/download_data.sh`). Useful to (i) cross-check your vetting
and (ii) build the barred fraction vs mass/environment for the whole sample.

In [ ]:
from bartw.morphology import read_gz_desi, crossmatch, select_barred
gz_path = data_dir("raw") / "gz_desi_deep_learning_catalog_friendly.parquet"
if gz_path.exists():
    gz = read_gz_desi(gz_path)
    m = crossmatch(sf, gz, cfg["galaxy_zoo_desi"]["match_radius_arcsec"])
    barred_gz, steps = select_barred(m, cfg["galaxy_zoo_desi"])
    print(steps); print("GZ-barred dwarfs at z<0.012:", int(barred_gz.Z_TIER.str.startswith("A").sum()))
    if len(has):
        chk = has.merge(barred_gz[["TARGETID", "GZ_BAR_FRACTION", "BAR_CLASS"]], on="TARGETID", how="left")
        display(chk[["TARGETID", "BARRED_SGA", "BAR_ELLIP", "GZ_BAR_FRACTION", "BAR_CLASS"] + (["VET"] if "VET" in chk else [])])
else:
    print("GZ DESI catalogue not found at", gz_path, "- see scripts/download_data.sh")

## 5. Disc and bar geometry, TW suitability

For the vetted `y`/`w` galaxies: r-band Legacy Surveys cutout sized to the
galaxy (4 x R25 on a side, R25 from the SGA or the Tractor model in the paper
table), isophote fit with the centre fixed, failed/sky-level isophotes
dropped. Disc PA and ellipticity are medians over 0.5-1.0 R25 (grey band);
the bar is the ellipticity peak inside 0.6 R25 that is resolved, more
elongated than the disc, and keeps its PA over >= 3 isophotes. TW needs
25 < i < 70 deg and 10 < |PA_bar - PA_disc| < 80 deg, and a disc PA error
of a few degrees at most: a PA_DISC_ERR above ~5 deg means the outer
isophotes are too noisy or irregular for TW whatever the flag says.

Red dashed ellipse = disc at R25; green = bar. Check each profile by eye: the
automatic a_bar can latch onto a bright HII region in a dwarf, and the
`status` column says why a bar was *not* measured.

In [ ]:
from bartw.photometry import fetch_cutout, cutout_size_for, measure_geometry, plot_geometry
from bartw.geometry import tw_suitable, pa_difference

def r25_arcsec(row, floor=8.0):
    """Outer radius for the isophote fit: SGA R25 if available, else the paper's
    Tractor-model D25/2, never below 3 x the Tractor half-light radius or ``floor``
    (the Tractor D25 underestimates low-surface-brightness dwarfs badly)."""
    cands = [floor]
    if "shape_r" in row and np.isfinite(row["shape_r"]) and row["shape_r"] > 0:
        cands.append(3.0 * float(row["shape_r"]))
    if "sma_sb25" in row and np.isfinite(row["sma_sb25"]) and row["sma_sb25"] > 0:
        cands.append(float(row["sma_sb25"]))
    elif "d25_adopted_arcsec" in row and np.isfinite(row["d25_adopted_arcsec"]):
        cands.append(float(row["d25_adopted_arcsec"]) / 2)
    return max(cands)

def geometry_for(row, show=True):
    g = cfg["geometry"]; pix = g["cutout_pixscale_arcsec"]
    r25 = r25_arcsec(row)
    img, hdr = fetch_cutout(row.RA, row.DEC, g["cutout_layer"], pix, cutout_size_for(r25, pix))
    res = measure_geometry(img, pix, r25)
    if show:
        plot_geometry(img, res, pix, title=f"{int(row.TARGETID)}  r band  R25={r25:.1f}\"")
    return dict(TARGETID=int(row.TARGETID), R25_ARCSEC=r25, PA_DISC=res["pa_disc"], PA_DISC_ERR=res["pa_disc_err"],
                EPS_DISC=res["eps_disc"], INCL_DEG=float(inclination_from_axis_ratio(1 - res["eps_disc"], g["q0"])),
                PA_BAR=res["pa_bar"], A_BAR_ARCSEC=res["a_bar"], A_BAR_ERR_ARCSEC=res["a_bar_err"], EPS_BAR=res["eps_bar"],
                DPA_DEG=float(pa_difference(res["pa_bar"], res["pa_disc"])) if np.isfinite(res["pa_bar"]) else np.nan,
                GEOM_STATUS=res["status"])

from bartw.geometry import inclination_from_axis_ratio
todo = tw_targets if len(tw_targets) else has
print(len(todo), "galaxies to measure", "(vetted y/w)" if len(tw_targets) else "(nothing vetted yet: all MUSE-covered)")

In [ ]:
geo = []
for _, r in todo.iterrows():
    try:
        geo.append(geometry_for(r, show=len(todo) <= 15))
    except Exception as exc:
        print(int(r.TARGETID), "failed:", exc)
geo = pd.DataFrame(geo)
if len(geo):
    ok, dpa, why = tw_suitable(geo.INCL_DEG, geo.PA_BAR, geo.PA_DISC, cfg["geometry"])
    geo["TW_SUITABLE"] = ok & (geo.PA_DISC_ERR <= 5.0)
    reason = np.where(ok & (geo.PA_DISC_ERR > 5.0), "disc PA error > 5 deg", why)
    # a missing disc or bar is a measurement outcome, not a geometric exclusion
    reason = np.where(geo.GEOM_STATUS != "ok", "no bar: " + geo.GEOM_STATUS.astype(str), reason)
    geo["TW_REASON"] = reason
    out = todo.merge(geo, on="TARGETID", suffixes=("_cat", ""))
    out["A_BAR_KPC"] = out.A_BAR_ARCSEC * out.kpc_per_arcsec
    out.to_csv(PROD / "sample_geometry.csv", index=False)
    print("TW-suitable:", int(out.TW_SUITABLE.sum()), "/", len(out)); print(out.TW_REASON.value_counts().to_string())
    display(out[["TARGETID", "Z", "R25_ARCSEC", "INCL_DEG_cat", "INCL_DEG", "PA_DISC", "PA_DISC_ERR", "PA_BAR", "DPA_DEG",
                 "A_BAR_ARCSEC", "A_BAR_KPC", "EPS_BAR", "GEOM_STATUS", "TW_SUITABLE", "TW_REASON"]].round(2))

## 6. What a TW measurement looks like (offline demo)

Synthetic barred disc with a known pattern speed through the same code that
`scripts/06_run_tw.py` uses on real kinematic maps. The right panel is the
diagnostic for every real galaxy: points on a straight line through the
origin, errors dominated by the disc PA.

In [ ]:
from bartw.mock import MockParams, make_mock
from bartw.tw import measure_pattern_speed, default_slit_centers, rotation_rate
from bartw.plotting import slit_overlay, tw_diagram

p = MockParams(psf_fwhm_arcsec=0.8, flux_noise_frac=0.05, vel_noise_kms=5.0, omega_p=30.0)
flux, vel, verr, t = make_mock(p)
dpa = pa_difference(t["bar_pa_sky"], p.pa_deg)
yc = default_slit_centers(t["a_bar_sky_arcsec"], dpa, cfg["tw"]["n_slits"], cfg["tw"]["slit_width_arcsec"])
res = measure_pattern_speed(flux, vel, p.pixscale, t["x0"], t["y0"], p.pa_deg, p.incl_deg, yc, cfg["tw"]["slit_width_arcsec"],
                            x_half_length_arcsec=1.5 * t["a_bar_sky_arcsec"], vel_err=verr, distance_mpc=p.distance_mpc,
                            pa_err_deg=cfg["tw"]["pa_error_deg"], n_mc=200, rng=0)
print("input Omega_p =", p.omega_p, "km/s/kpc ->", res.summary())
rr = rotation_rate(res.omega_p, res.omega_p_err, vc=p.vflat, vc_err=5, a_bar_kpc=p.a_bar_arcsec * t["scale_kpc_arcsec"], a_bar_err_kpc=0.1, rng=0)
print(f"R = {rr['R']:.2f} (+{rr['R_err_hi']:.2f}/-{rr['R_err_lo']:.2f}) -> {rr['regime']}  p_slow={rr['p_slow']:.2f} p_fast={rr['p_fast']:.2f} p_uf={rr['p_ultrafast']:.2f}")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
slit_overlay(flux, vel, res, cfg["tw"]["slit_width_arcsec"], p.pixscale, True, axes[:2], 1.5 * t["a_bar_sky_arcsec"])
tw_diagram(res, axes[2], title="synthetic disc"); plt.tight_layout()

## 7. Hand-over to the scripts

* Download the cubes (`dp_id`s from section 2) from https://archive.eso.org; proprietary ones need the PI's permission.
* Kinematics: `python scripts/05_stellar_kinematics.py CUBE.fits --z Z --templates spectra_emiles_9.0.npz --out data/products/kin_NAME.fits`
* TW: `python scripts/06_run_tw.py data/products/kin_NAME.fits --name NAME --pa PA_DISC --pa-err PA_DISC_ERR --incl INCL_DEG --bar-pa PA_BAR --a-bar A_BAR_ARCSEC --distance D_MPC [--vc VC --vc-err DVC]`
  with the numbers from `products/sample_geometry.csv`.
* Benchmark first on NGC 4516 / IC 3167 against Cuomo et al. (2022, 2024).